# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedfurqan1/FlyRank-MachineLearning/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Freshness Multiplier

The paper observed that recently refreshed mature pages had substantially higher health scores and impressions than stale mature pages. I would ask whether refresh dates occurred before the measured performance window and whether refreshed and stale pages were comparable in prior visibility, client mix, topic, and historical performance. These checks would clarify whether the result measures a directional association with refresh activity or partly reflects which pages were selected for refresh.

### Finding 2 — Predicting growth in the ML appendix

The appendix reports 71% holdout accuracy when separating growing from declining content. I would ask whether the label comes directly from the disclosed 30-day-versus-previous-30-day impression trend, what the outcome base rate was, and whether the 80/20 split kept each brand entirely within either training or validation. A brand-grouped or time-aware evaluation would better show whether the measured performance carries to unseen brands or a later period.

---



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Locked evaluation design

March 2026 supplies all model features and April 2026 supplies the observed decline label. The primary metric is Precision@50, with Precision@20 as a secondary capacity check. Models use fixed features, deterministic row ordering, and seed 42.

The audit will compare a deliberately naive row-random split with five-fold validation grouped by client. The random result is only a diagnostic; the grouped result is the honest estimate for unseen clients. June remains sealed.

In [1]:
import getpass

import duckdb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import GroupKFold

RANDOM_SEED = 42
N_SPLITS = 5
PRIMARY_K = 50
SECONDARY_K = 20
DECLINE_THRESHOLD_FRACTION = -0.15
OUTCOME_MULTIPLIER = 1 + DECLINE_THRESHOLD_FRACTION
BASELINE_MIN_IMPRESSIONS_COUNT = 100
BASELINE_MAX_AVG_SEARCH_POSITION = 20
SEALED_TEST_MONTH = "2026-06"
LOADED_MONTHS = ("2026-03", "2026-04")

MODEL_FEATURE_COLUMNS = [
    "log1p_march_impressions_count",
    "log1p_march_clicks_count",
    "march_avg_search_position",
    "march_ctr_pct",
    "is_zero_click",
    "is_baseline_eligible",
]

FORBIDDEN_FEATURE_COLUMNS = {
    "client_hash_id",
    "content_hash_id",
    "april_impressions_count",
    "april_impressions_change_pct",
    "is_declining_label",
    "baseline_priority_impressions_count",
    "trend_direction",
    "trend_pct",
}

assert SEALED_TEST_MONTH not in LOADED_MONTHS
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(FORBIDDEN_FEATURE_COLUMNS)

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

if not hf_token or not hf_token.startswith("hf_"):
    raise ValueError("A valid Hugging Face READ token is required.")

connection = duckdb.connect()
safe_token = hf_token.replace("'", "''")

connection.execute(
    f"""
    CREATE OR REPLACE SECRET flyrank_hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

del hf_token, safe_token

warehouse_root = "hf://datasets/FlyRank/internship-warehouse"
march_fact_path = (
    f"{warehouse_root}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)
april_fact_path = (
    f"{warehouse_root}/fact_content_daily_performance/"
    "month=2026-04/*.parquet"
)

analysis_df = connection.execute(
    f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions_count,
            SUM(gsc_clicks) AS march_clicks_count,
            SUM(gsc_sum_position)
                / NULLIF(SUM(gsc_impressions), 0)
                AS march_avg_search_position
        FROM read_parquet('{march_fact_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions_count
        FROM read_parquet('{april_fact_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        march.client_hash_id,
        march.content_hash_id,
        march.march_impressions_count,
        march.march_clicks_count,
        march.march_avg_search_position,
        100.0 * march.march_clicks_count
            / NULLIF(march.march_impressions_count, 0)
            AS march_ctr_pct,
        april.april_impressions_count,
        CASE
            WHEN april.april_impressions_count
                < {OUTCOME_MULTIPLIER}
                * march.march_impressions_count
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march
    INNER JOIN april
        USING (client_hash_id, content_hash_id)
    WHERE march.march_impressions_count > 0
    """
).df()

connection.close()

analysis_df = (
    analysis_df
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)

assert not analysis_df.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()
assert analysis_df["march_impressions_count"].gt(0).all()
assert analysis_df["is_declining_label"].isin([0, 1]).all()

grouped_splitter = GroupKFold(n_splits=N_SPLITS)
analysis_df["grouped_validation_fold"] = -1

for fold_number, (_, validation_index) in enumerate(
    grouped_splitter.split(
        X=np.zeros(len(analysis_df)),
        y=analysis_df["is_declining_label"],
        groups=analysis_df["client_hash_id"],
    ),
    start=1,
):
    analysis_df.loc[
        analysis_df.iloc[validation_index].index,
        "grouped_validation_fold",
    ] = fold_number

assert analysis_df["grouped_validation_fold"].between(
    1, N_SPLITS
).all()

client_fold_count = (
    analysis_df
    .groupby("client_hash_id")["grouped_validation_fold"]
    .nunique()
)
assert client_fold_count.eq(1).all()

for fold_number in range(1, N_SPLITS + 1):
    training_client_ids = set(
        analysis_df.loc[
            analysis_df["grouped_validation_fold"] != fold_number,
            "client_hash_id",
        ]
    )
    validation_client_ids = set(
        analysis_df.loc[
            analysis_df["grouped_validation_fold"] == fold_number,
            "client_hash_id",
        ]
    )
    assert training_client_ids.isdisjoint(validation_client_ids)

grouped_fold_summary = (
    analysis_df
    .groupby("grouped_validation_fold")
    .agg(
        validation_row_count=("content_hash_id", "size"),
        validation_client_count=("client_hash_id", "nunique"),
        decline_base_rate_fraction=("is_declining_label", "mean"),
    )
    .reset_index()
)

grouped_fold_summary["decline_base_rate_pct"] = (
    100 * grouped_fold_summary["decline_base_rate_fraction"]
).round(1)

grouped_fold_summary = grouped_fold_summary[
    [
        "grouped_validation_fold",
        "validation_row_count",
        "validation_client_count",
        "decline_base_rate_pct",
    ]
]

fold_size_ratio = (
    grouped_fold_summary["validation_row_count"].max()
    / grouped_fold_summary["validation_row_count"].min()
)

print(f"Modeling row count: {len(analysis_df):,}")
print(
    "Client count: "
    f"{analysis_df['client_hash_id'].nunique():,}"
)
print(
    "Overall decline base rate: "
    f"{analysis_df['is_declining_label'].mean():.1%}"
)
print(f"Primary metric: Precision@{PRIMARY_K}")
print(f"Secondary metric: Precision@{SECONDARY_K}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Largest-to-smallest fold ratio: {fold_size_ratio:.2f}")
print("Client overlap check: passed")
print(f"{SEALED_TEST_MONTH} loaded: no")
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Locked features: {MODEL_FEATURE_COLUMNS}")

display(grouped_fold_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling row count: 158,549
Client count: 46
Overall decline base rate: 51.1%
Primary metric: Precision@50
Secondary metric: Precision@20
Random seed: 42
Largest-to-smallest fold ratio: 1.00
Client overlap check: passed
2026-06 loaded: no
pandas version: 2.2.3
NumPy version: 2.1.3
DuckDB version: 1.3.2
scikit-learn version: 1.6.1
Locked features: ['log1p_march_impressions_count', 'log1p_march_clicks_count', 'march_avg_search_position', 'march_ctr_pct', 'is_zero_click', 'is_baseline_eligible']


,grouped_validation_fold,validation_row_count,validation_client_count,decline_base_rate_pct
0,1,31712,8,46.3
1,2,31708,8,67.8
2,3,31707,10,35.6
3,4,31706,11,54.4
4,5,31716,9,51.3


### Before and after validation

The “before” design randomly distributes rows and therefore allows the same clients to appear in training and validation. The “after” design keeps clients intact. All methods use the same validation rows within each design, so differences between methods are directly comparable.

Scores are used only to rank candidates. They are not interpreted as calibrated probabilities.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

analysis_df["log1p_march_impressions_count"] = np.log1p(
    analysis_df["march_impressions_count"]
)
analysis_df["log1p_march_clicks_count"] = np.log1p(
    analysis_df["march_clicks_count"]
)
analysis_df["is_zero_click"] = analysis_df[
    "march_clicks_count"
].eq(0)
analysis_df["is_baseline_eligible"] = (
    analysis_df["march_impressions_count"].ge(
        BASELINE_MIN_IMPRESSIONS_COUNT
    )
    & analysis_df["is_zero_click"]
    & analysis_df["march_avg_search_position"].gt(0)
    & analysis_df["march_avg_search_position"].le(
        BASELINE_MAX_AVG_SEARCH_POSITION
    )
)
analysis_df["baseline_priority_impressions_count"] = np.where(
    analysis_df["is_baseline_eligible"],
    analysis_df["march_impressions_count"],
    0,
)

assert set(MODEL_FEATURE_COLUMNS).issubset(analysis_df.columns)
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(
    FORBIDDEN_FEATURE_COLUMNS
)
assert analysis_df[MODEL_FEATURE_COLUMNS].notna().all().all()

random_splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)
analysis_df["random_validation_fold"] = -1

for fold_number, (_, validation_index) in enumerate(
    random_splitter.split(
        X=np.zeros(len(analysis_df)),
        y=analysis_df["is_declining_label"],
    ),
    start=1,
):
    analysis_df.loc[
        analysis_df.iloc[validation_index].index,
        "random_validation_fold",
    ] = fold_number

assert analysis_df["random_validation_fold"].between(
    1, N_SPLITS
).all()

estimators = {
    "Logistic Regression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=150,
                    max_depth=8,
                    min_samples_leaf=50,
                    max_features="sqrt",
                    n_jobs=-1,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    ),
}

def precision_at_k(frame, ranking_column, k):
    ranked_frame = frame.sort_values(
        [ranking_column, "content_hash_id"],
        ascending=[False, True],
        kind="mergesort",
    )
    return ranked_frame.head(k)["is_declining_label"].mean()

split_designs = {
    "row_random_before": "random_validation_fold",
    "client_grouped_after": "grouped_validation_fold",
}

fold_metric_rows = []
oof_prediction_frames = []

for split_design, fold_column in split_designs.items():
    for fold_number in range(1, N_SPLITS + 1):
        training_mask = analysis_df[fold_column].ne(fold_number)
        validation_mask = analysis_df[fold_column].eq(fold_number)

        training_client_ids = set(
            analysis_df.loc[training_mask, "client_hash_id"]
        )
        validation_client_ids = set(
            analysis_df.loc[validation_mask, "client_hash_id"]
        )
        client_overlap_count = len(
            training_client_ids.intersection(validation_client_ids)
        )

        X_train = analysis_df.loc[
            training_mask, MODEL_FEATURE_COLUMNS
        ]
        y_train = analysis_df.loc[
            training_mask, "is_declining_label"
        ]
        X_validation = analysis_df.loc[
            validation_mask, MODEL_FEATURE_COLUMNS
        ]

        validation_frame = analysis_df.loc[
            validation_mask,
            [
                "client_hash_id",
                "content_hash_id",
                "march_impressions_count",
                "march_clicks_count",
                "march_avg_search_position",
                "march_ctr_pct",
                "april_impressions_count",
                "is_declining_label",
                "is_baseline_eligible",
                "baseline_priority_impressions_count",
            ],
        ].copy()

        assert len(validation_frame) >= PRIMARY_K
        assert validation_frame[
            "is_baseline_eligible"
        ].sum() >= PRIMARY_K
        assert validation_frame[
            "is_declining_label"
        ].nunique() == 2

        for model_name, estimator in estimators.items():
            estimator.fit(X_train, y_train)
            ranking_score = estimator.predict_proba(
                X_validation
            )[:, 1]

            if model_name == "Logistic Regression":
                score_column = "logistic_ranking_score_0_1"
            else:
                score_column = "forest_ranking_score_0_1"

            validation_frame[score_column] = ranking_score

        validation_frame["split_design"] = split_design
        validation_frame["validation_fold"] = fold_number
        oof_prediction_frames.append(validation_frame)

        fold_metric_rows.append(
            {
                "split_design": split_design,
                "validation_fold": fold_number,
                "validation_row_count": len(validation_frame),
                "validation_client_count": len(
                    validation_client_ids
                ),
                "train_validation_client_overlap_count": (
                    client_overlap_count
                ),
                "decline_base_rate_pct": (
                    100
                    * validation_frame[
                        "is_declining_label"
                    ].mean()
                ),
                "baseline_p20_pct": (
                    100
                    * precision_at_k(
                        validation_frame,
                        "baseline_priority_impressions_count",
                        SECONDARY_K,
                    )
                ),
                "baseline_p50_pct": (
                    100
                    * precision_at_k(
                        validation_frame,
                        "baseline_priority_impressions_count",
                        PRIMARY_K,
                    )
                ),
                "logistic_p20_pct": (
                    100
                    * precision_at_k(
                        validation_frame,
                        "logistic_ranking_score_0_1",
                        SECONDARY_K,
                    )
                ),
                "logistic_p50_pct": (
                    100
                    * precision_at_k(
                        validation_frame,
                        "logistic_ranking_score_0_1",
                        PRIMARY_K,
                    )
                ),
                "forest_p20_pct": (
                    100
                    * precision_at_k(
                        validation_frame,
                        "forest_ranking_score_0_1",
                        SECONDARY_K,
                    )
                ),
                "forest_p50_pct": (
                    100
                    * precision_at_k(
                        validation_frame,
                        "forest_ranking_score_0_1",
                        PRIMARY_K,
                    )
                ),
            }
        )

fold_metrics = pd.DataFrame(fold_metric_rows)
metric_columns = [
    "decline_base_rate_pct",
    "baseline_p20_pct",
    "baseline_p50_pct",
    "logistic_p20_pct",
    "logistic_p50_pct",
    "forest_p20_pct",
    "forest_p50_pct",
]
fold_metrics[metric_columns] = fold_metrics[
    metric_columns
].round(1)

oof_predictions = pd.concat(
    oof_prediction_frames,
    ignore_index=True,
)

assert len(fold_metrics) == 2 * N_SPLITS
assert len(oof_predictions) == 2 * len(analysis_df)
assert fold_metrics.loc[
    fold_metrics["split_design"].eq("client_grouped_after"),
    "train_validation_client_overlap_count",
].eq(0).all()
assert fold_metrics.loc[
    fold_metrics["split_design"].eq("row_random_before"),
    "train_validation_client_overlap_count",
].gt(0).all()

display(fold_metrics)

,split_design,validation_fold,validation_row_count,validation_client_count,train_validation_client_overlap_count,decline_base_rate_pct,baseline_p20_pct,baseline_p50_pct,logistic_p20_pct,logistic_p50_pct,forest_p20_pct,forest_p50_pct
0,row_random_before,1,31710,43,43,51.1,65.0,70.0,75.0,66.0,60.0,74.0
1,row_random_before,2,31710,44,43,51.1,75.0,64.0,70.0,70.0,80.0,78.0
2,row_random_before,3,31710,45,45,51.1,70.0,68.0,80.0,74.0,70.0,72.0
3,row_random_before,4,31710,44,44,51.1,80.0,78.0,85.0,70.0,75.0,78.0
4,row_random_before,5,31709,43,43,51.1,80.0,80.0,75.0,78.0,85.0,78.0
5,client_grouped_after,1,31712,8,0,46.3,65.0,56.0,65.0,58.0,65.0,66.0
6,client_grouped_after,2,31708,8,0,67.8,80.0,84.0,80.0,84.0,85.0,84.0
7,client_grouped_after,3,31707,10,0,35.6,70.0,60.0,75.0,68.0,50.0,54.0
8,client_grouped_after,4,31706,11,0,54.4,95.0,78.0,75.0,80.0,40.0,56.0
9,client_grouped_after,5,31716,9,0,51.3,65.0,66.0,65.0,56.0,70.0,68.0


In [3]:
method_metric_prefixes = {
    "Week 4 baseline": "baseline",
    "Logistic Regression": "logistic",
    "Random Forest": "forest",
}

summary_rows = []

for method_name, metric_prefix in method_metric_prefixes.items():
    random_rows = fold_metrics[
        fold_metrics["split_design"].eq("row_random_before")
    ]
    grouped_rows = fold_metrics[
        fold_metrics["split_design"].eq("client_grouped_after")
    ]

    random_p50_pct = random_rows[
        f"{metric_prefix}_p50_pct"
    ].mean()
    grouped_p50_pct = grouped_rows[
        f"{metric_prefix}_p50_pct"
    ].mean()
    random_p20_pct = random_rows[
        f"{metric_prefix}_p20_pct"
    ].mean()
    grouped_p20_pct = grouped_rows[
        f"{metric_prefix}_p20_pct"
    ].mean()

    summary_rows.append(
        {
            "method": method_name,
            "outcome_base_rate_pct": (
                100
                * analysis_df["is_declining_label"].mean()
            ),
            "random_mean_p50_pct": random_p50_pct,
            "grouped_mean_p50_pct": grouped_p50_pct,
            "p50_after_minus_before_pp": (
                grouped_p50_pct - random_p50_pct
            ),
            "grouped_p50_std_pp": grouped_rows[
                f"{metric_prefix}_p50_pct"
            ].std(ddof=1),
            "random_mean_p20_pct": random_p20_pct,
            "grouped_mean_p20_pct": grouped_p20_pct,
            "p20_after_minus_before_pp": (
                grouped_p20_pct - random_p20_pct
            ),
            "grouped_p20_std_pp": grouped_rows[
                f"{metric_prefix}_p20_pct"
            ].std(ddof=1),
        }
    )

before_after_summary = pd.DataFrame(summary_rows)

numeric_summary_columns = before_after_summary.select_dtypes(
    include="number"
).columns
before_after_summary[numeric_summary_columns] = (
    before_after_summary[numeric_summary_columns].round(1)
)

assert before_after_summary["outcome_base_rate_pct"].eq(
    round(
        100 * analysis_df["is_declining_label"].mean(),
        1,
    )
).all()

display(before_after_summary)


,method,outcome_base_rate_pct,random_mean_p50_pct,grouped_mean_p50_pct,p50_after_minus_before_pp,grouped_p50_std_pp,random_mean_p20_pct,grouped_mean_p20_pct,p20_after_minus_before_pp,grouped_p20_std_pp
0,Week 4 baseline,51.1,72.0,68.8,-3.2,11.9,74.0,75.0,1.0,12.7
1,Logistic Regression,51.1,71.6,69.2,-2.4,12.6,77.0,72.0,-5.0,6.7
2,Random Forest,51.1,76.0,65.6,-10.4,11.9,74.0,62.0,-12.0,17.5


### Error examples

Error analysis uses the honest client-grouped predictions and the primary top-50 capacity. A weak pick is a recommended item whose April impression change did not cross the predefined decline threshold.

The examples are selected to show different failure modes. Identifiers are used internally for deterministic ranking but are removed before display.

In [4]:
grouped_oof = oof_predictions.loc[
    oof_predictions["split_design"].eq("client_grouped_after")
].copy()

grouped_oof = grouped_oof.sort_values(
    [
        "validation_fold",
        "logistic_ranking_score_0_1",
        "content_hash_id",
    ],
    ascending=[True, False, True],
    kind="mergesort",
)

grouped_oof["logistic_rank"] = (
    grouped_oof.groupby("validation_fold").cumcount() + 1
)

grouped_top50 = grouped_oof.loc[
    grouped_oof["logistic_rank"].le(PRIMARY_K)
].copy()

grouped_top50["april_impressions_change_pct"] = (
    100
    * (
        grouped_top50["april_impressions_count"]
        - grouped_top50["march_impressions_count"]
    )
    / grouped_top50["march_impressions_count"]
)

grouped_top50["decline_threshold_pct"] = (
    100 * DECLINE_THRESHOLD_FRACTION
)
grouped_top50["threshold_gap_pp"] = (
    grouped_top50["april_impressions_change_pct"]
    - grouped_top50["decline_threshold_pct"]
)

correct_pick_count = int(
    grouped_top50["is_declining_label"].sum()
)
weak_pick_count = int(
    grouped_top50["is_declining_label"].eq(0).sum()
)
total_pick_count = len(grouped_top50)

assert total_pick_count == N_SPLITS * PRIMARY_K
assert correct_pick_count + weak_pick_count == total_pick_count
assert (
    grouped_top50.groupby("validation_fold").size()
    .eq(PRIMARY_K)
    .all()
)

weak_picks = grouped_top50.loc[
    grouped_top50["is_declining_label"].eq(0)
].copy()

highest_ranked_weak_pick = (
    weak_picks.sort_values(
        [
            "logistic_rank",
            "validation_fold",
            "content_hash_id",
        ],
        ascending=[True, True, True],
        kind="mergesort",
    )
    .iloc[[0]]
    .copy()
)
highest_ranked_weak_pick["example_type"] = (
    "highest_ranked_weak_pick"
)

remaining_weak_picks = weak_picks.drop(
    index=highest_ranked_weak_pick.index
)

closest_to_threshold = (
    remaining_weak_picks.sort_values(
        [
            "threshold_gap_pp",
            "logistic_rank",
            "validation_fold",
            "content_hash_id",
        ],
        ascending=[True, True, True, True],
        kind="mergesort",
    )
    .iloc[[0]]
    .copy()
)
closest_to_threshold["example_type"] = (
    "closest_to_threshold"
)

remaining_weak_picks = remaining_weak_picks.drop(
    index=closest_to_threshold.index
)

largest_april_increase = (
    remaining_weak_picks.sort_values(
        [
            "april_impressions_change_pct",
            "logistic_rank",
            "validation_fold",
            "content_hash_id",
        ],
        ascending=[False, True, True, True],
        kind="mergesort",
    )
    .iloc[[0]]
    .copy()
)
largest_april_increase["example_type"] = (
    "largest_april_increase"
)

error_examples = pd.concat(
    [
        highest_ranked_weak_pick,
        closest_to_threshold,
        largest_april_increase,
    ],
    ignore_index=True,
)

error_examples = error_examples[
    [
        "example_type",
        "validation_fold",
        "logistic_rank",
        "logistic_ranking_score_0_1",
        "march_impressions_count",
        "march_clicks_count",
        "march_avg_search_position",
        "march_ctr_pct",
        "is_baseline_eligible",
        "april_impressions_change_pct",
        "decline_threshold_pct",
        "threshold_gap_pp",
    ]
].copy()

rounding_rules = {
    "logistic_ranking_score_0_1": 3,
    "march_avg_search_position": 2,
    "march_ctr_pct": 3,
    "april_impressions_change_pct": 1,
    "decline_threshold_pct": 1,
    "threshold_gap_pp": 1,
}
error_examples = error_examples.round(rounding_rules)

assert len(error_examples) == 3
assert error_examples["threshold_gap_pp"].ge(0).all()

print(f"Grouped top-50 pick count: {total_pick_count}")
print(f"Correct pick count: {correct_pick_count}")
print(f"Weak pick count: {weak_pick_count}")
print(
    "Observed grouped top-50 precision: "
    f"{100 * correct_pick_count / total_pick_count:.1f}%"
)

display(error_examples)

Grouped top-50 pick count: 250
Correct pick count: 173
Weak pick count: 77
Observed grouped top-50 precision: 69.2%


,example_type,validation_fold,logistic_rank,logistic_ranking_score_0_1,march_impressions_count,march_clicks_count,march_avg_search_position,march_ctr_pct,is_baseline_eligible,april_impressions_change_pct,decline_threshold_pct,threshold_gap_pp
0,highest_ranked_weak_pick,1,1,0.896,134984.0,1.0,2.69,0.001,False,-5.1,-15.0,9.9
1,closest_to_threshold,2,9,0.771,8258.0,0.0,5.47,0.000,True,-14.3,-15.0,0.7
2,largest_april_increase,5,19,0.836,6143.0,0.0,9.95,0.000,True,4262.0,-15.0,4277.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Feature and timeline audit

Every model input must be available at the end of March, before the April outcome is observed. Identifiers remain available only for grouping and deterministic ordering, while April fields, label siblings, product scores, and the sealed period are excluded.

The baseline-eligibility indicator is reconstructed from legal March inputs rather than imported from a product. It is not label leakage, but its decision-rule origin warrants a separate sensitivity check.

In [5]:
expected_march_ctr_pct = (
    100
    * analysis_df["march_clicks_count"]
    / analysis_df["march_impressions_count"]
)
expected_is_zero_click = analysis_df[
    "march_clicks_count"
].eq(0)
expected_is_baseline_eligible = (
    analysis_df["march_impressions_count"].ge(
        BASELINE_MIN_IMPRESSIONS_COUNT
    )
    & expected_is_zero_click
    & analysis_df["march_avg_search_position"].gt(0)
    & analysis_df["march_avg_search_position"].le(
        BASELINE_MAX_AVG_SEARCH_POSITION
    )
)

assert np.allclose(
    analysis_df["march_ctr_pct"],
    expected_march_ctr_pct,
)
assert analysis_df["is_zero_click"].equals(
    expected_is_zero_click
)
assert analysis_df["is_baseline_eligible"].equals(
    expected_is_baseline_eligible
)
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(
    {
        "client_hash_id",
        "content_hash_id",
        "april_impressions_count",
        "is_declining_label",
        "baseline_priority_impressions_count",
        "trend_direction",
        "trend_pct",
    }
)
assert SEALED_TEST_MONTH not in LOADED_MONTHS

feature_audit = pd.DataFrame(
    [
        {
            "feature_or_field": "log1p_march_impressions_count",
            "role": "model_feature",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "allowed",
        },
        {
            "feature_or_field": "log1p_march_clicks_count",
            "role": "model_feature",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "allowed",
        },
        {
            "feature_or_field": "march_avg_search_position",
            "role": "model_feature",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "allowed",
        },
        {
            "feature_or_field": "march_ctr_pct",
            "role": "model_feature",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "allowed",
        },
        {
            "feature_or_field": "is_zero_click",
            "role": "model_feature",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "allowed",
        },
        {
            "feature_or_field": "is_baseline_eligible",
            "role": "model_feature",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": True,
            "audit_verdict": "sensitivity_check_required",
        },
        {
            "feature_or_field": "client_hash_id",
            "role": "grouping_context",
            "source_window": "entity metadata",
            "is_available_at_prediction": True,
            "is_identifier": True,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "excluded_from_features",
        },
        {
            "feature_or_field": "content_hash_id",
            "role": "ordering_context",
            "source_window": "entity metadata",
            "is_available_at_prediction": True,
            "is_identifier": True,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "excluded_from_features",
        },
        {
            "feature_or_field": "april_impressions_count",
            "role": "outcome_input",
            "source_window": "April 2026",
            "is_available_at_prediction": False,
            "is_identifier": False,
            "is_label_or_future_derived": True,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "excluded_future",
        },
        {
            "feature_or_field": "is_declining_label",
            "role": "label",
            "source_window": "April versus March",
            "is_available_at_prediction": False,
            "is_identifier": False,
            "is_label_or_future_derived": True,
            "is_product_generated": False,
            "is_decision_rule_derived": False,
            "audit_verdict": "excluded_label",
        },
        {
            "feature_or_field": "baseline_priority_impressions_count",
            "role": "baseline_ranking_field",
            "source_window": "March 2026",
            "is_available_at_prediction": True,
            "is_identifier": False,
            "is_label_or_future_derived": False,
            "is_product_generated": False,
            "is_decision_rule_derived": True,
            "audit_verdict": "baseline_only",
        },
        {
            "feature_or_field": "trend_direction_or_trend_pct",
            "role": "prohibited_label_sibling",
            "source_window": "overlapping trend window",
            "is_available_at_prediction": False,
            "is_identifier": False,
            "is_label_or_future_derived": True,
            "is_product_generated": True,
            "is_decision_rule_derived": False,
            "audit_verdict": "excluded_leakage",
        },
    ]
)

assert set(MODEL_FEATURE_COLUMNS) == set(
    feature_audit.loc[
        feature_audit["role"].eq("model_feature"),
        "feature_or_field",
    ]
)
assert not feature_audit.loc[
    feature_audit["role"].eq("model_feature"),
    "is_label_or_future_derived",
].any()
assert not feature_audit.loc[
    feature_audit["role"].eq("model_feature"),
    "is_identifier",
].any()
assert not feature_audit.loc[
    feature_audit["role"].eq("model_feature"),
    "is_product_generated",
].any()

display(feature_audit)

,feature_or_field,role,source_window,is_available_at_prediction,is_identifier,is_label_or_future_derived,is_product_generated,is_decision_rule_derived,audit_verdict
0,log1p_march_impressions_count,model_feature,March 2026,True,False,False,False,False,allowed
1,log1p_march_clicks_count,model_feature,March 2026,True,False,False,False,False,allowed
2,march_avg_search_position,model_feature,March 2026,True,False,False,False,False,allowed
3,march_ctr_pct,model_feature,March 2026,True,False,False,False,False,allowed
4,is_zero_click,model_feature,March 2026,True,False,False,False,False,allowed
5,is_baseline_eligible,model_feature,March 2026,True,False,False,False,True,sensitivity_check_required
6,client_hash_id,grouping_context,entity metadata,True,True,False,False,False,excluded_from_features
7,content_hash_id,ordering_context,entity metadata,True,True,False,False,False,excluded_from_features
8,april_impressions_count,outcome_input,April 2026,False,False,True,False,False,excluded_future
9,is_declining_label,label,April versus March,False,False,True,False,False,excluded_label


### Baseline-flag sensitivity and leakage harness

I rerun Logistic Regression without the baseline-derived indicator while preserving the grouped folds, rows, seed, and metrics. This tests whether the model depends materially on the earlier decision rule.

A separate deliberately invalid run supplies the outcome label as an input. It is included only to verify that the evaluation detects direct leakage and is never considered a candidate model.

In [6]:
SENSITIVITY_FEATURE_COLUMNS = [
    "log1p_march_impressions_count",
    "log1p_march_clicks_count",
    "march_avg_search_position",
    "march_ctr_pct",
    "is_zero_click",
]

LEAKY_HARNESS_FEATURE_COLUMNS = (
    SENSITIVITY_FEATURE_COLUMNS
    + ["is_declining_label"]
)

assert "is_baseline_eligible" not in (
    SENSITIVITY_FEATURE_COLUMNS
)
assert "is_declining_label" in (
    LEAKY_HARNESS_FEATURE_COLUMNS
)

audit_variant_specs = {
    "logistic_without_baseline_flag": (
        SENSITIVITY_FEATURE_COLUMNS
    ),
    "intentional_label_leak_harness": (
        LEAKY_HARNESS_FEATURE_COLUMNS
    ),
}

audit_fold_rows = []

for variant_name, feature_columns in audit_variant_specs.items():
    for fold_number in range(1, N_SPLITS + 1):
        training_mask = analysis_df[
            "grouped_validation_fold"
        ].ne(fold_number)
        validation_mask = analysis_df[
            "grouped_validation_fold"
        ].eq(fold_number)

        audit_estimator = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=2000,
                        random_state=RANDOM_SEED,
                    ),
                ),
            ]
        )

        audit_estimator.fit(
            analysis_df.loc[
                training_mask, feature_columns
            ],
            analysis_df.loc[
                training_mask, "is_declining_label"
            ],
        )

        audit_validation_frame = analysis_df.loc[
            validation_mask,
            ["content_hash_id", "is_declining_label"],
        ].copy()

        audit_validation_frame[
            "audit_ranking_score_0_1"
        ] = audit_estimator.predict_proba(
            analysis_df.loc[
                validation_mask, feature_columns
            ]
        )[:, 1]

        audit_fold_rows.append(
            {
                "variant": variant_name,
                "validation_fold": fold_number,
                "p20_pct": (
                    100
                    * precision_at_k(
                        audit_validation_frame,
                        "audit_ranking_score_0_1",
                        SECONDARY_K,
                    )
                ),
                "p50_pct": (
                    100
                    * precision_at_k(
                        audit_validation_frame,
                        "audit_ranking_score_0_1",
                        PRIMARY_K,
                    )
                ),
            }
        )

audit_variant_fold_metrics = pd.DataFrame(
    audit_fold_rows
)

grouped_reference_metrics = fold_metrics.loc[
    fold_metrics["split_design"].eq(
        "client_grouped_after"
    )
]

reference_rows = []

for variant_name, metric_prefix, status in [
    (
        "frozen_baseline_reference",
        "baseline",
        "valid_reference",
    ),
    (
        "week5_logistic_six_features",
        "logistic",
        "valid_reference",
    ),
]:
    reference_rows.append(
        {
            "variant": variant_name,
            "audit_status": status,
            "outcome_base_rate_pct": (
                100
                * analysis_df[
                    "is_declining_label"
                ].mean()
            ),
            "mean_p20_pct": grouped_reference_metrics[
                f"{metric_prefix}_p20_pct"
            ].mean(),
            "mean_p50_pct": grouped_reference_metrics[
                f"{metric_prefix}_p50_pct"
            ].mean(),
            "p50_std_pp": grouped_reference_metrics[
                f"{metric_prefix}_p50_pct"
            ].std(ddof=1),
        }
    )

audit_summary_rows = []

for variant_name, variant_rows in (
    audit_variant_fold_metrics.groupby(
        "variant",
        sort=False,
    )
):
    if variant_name == (
        "intentional_label_leak_harness"
    ):
        audit_status = "invalid_leakage_test"
    else:
        audit_status = "valid_sensitivity_test"

    audit_summary_rows.append(
        {
            "variant": variant_name,
            "audit_status": audit_status,
            "outcome_base_rate_pct": (
                100
                * analysis_df[
                    "is_declining_label"
                ].mean()
            ),
            "mean_p20_pct": variant_rows[
                "p20_pct"
            ].mean(),
            "mean_p50_pct": variant_rows[
                "p50_pct"
            ].mean(),
            "p50_std_pp": variant_rows[
                "p50_pct"
            ].std(ddof=1),
        }
    )

leakage_sensitivity_summary = pd.DataFrame(
    reference_rows + audit_summary_rows
)

week5_logistic_p50_pct = (
    leakage_sensitivity_summary.loc[
        leakage_sensitivity_summary["variant"].eq(
            "week5_logistic_six_features"
        ),
        "mean_p50_pct",
    ].iloc[0]
)

leakage_sensitivity_summary[
    "p50_difference_vs_week5_logistic_pp"
] = (
    leakage_sensitivity_summary["mean_p50_pct"]
    - week5_logistic_p50_pct
)

numeric_columns = (
    leakage_sensitivity_summary.select_dtypes(
        include="number"
    ).columns
)
leakage_sensitivity_summary[numeric_columns] = (
    leakage_sensitivity_summary[
        numeric_columns
    ].round(1)
)

leaky_harness_rows = audit_variant_fold_metrics.loc[
    audit_variant_fold_metrics["variant"].eq(
        "intentional_label_leak_harness"
    )
]

assert leaky_harness_rows["p20_pct"].eq(100).all()
assert leaky_harness_rows["p50_pct"].eq(100).all()

display(leakage_sensitivity_summary)

,variant,audit_status,outcome_base_rate_pct,mean_p20_pct,mean_p50_pct,p50_std_pp,p50_difference_vs_week5_logistic_pp
0,frozen_baseline_reference,valid_reference,51.1,75.0,68.8,11.9,-0.4
1,week5_logistic_six_features,valid_reference,51.1,72.0,69.2,12.6,0.0
2,logistic_without_baseline_flag,valid_sensitivity_test,51.1,72.0,69.2,12.6,0.0
3,intentional_label_leak_harness,invalid_leakage_test,51.1,100.0,100.0,0.0,30.8


### Leakage-audit verdict

All outcome, identifier, trend, product-score, and sealed-period fields were excluded from the model. The declared features were available by the end of March, and the intentional label-leak test reached 100% Precision@20 and Precision@50 as expected.

Removing the baseline-derived eligibility indicator left grouped Precision@20 and Precision@50 unchanged at 72.0% and 69.2%. The measured Week 5 ranking result therefore did not depend on that indicator at either operational capacity, although a future simplified model could omit it.

---



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

> “Machine learning beats fixed rules because search performance decay involves non-linear interactions across multiple signals.”

### Evidence-based rewrite

Across five client-grouped March-to-April validation folds, Logistic Regression measured 69.2% mean Precision@50, compared with 68.8% for the fixed baseline and a 51.1% outcome base rate. The 0.4 percentage-point difference was small relative to roughly 12 percentage points of fold variation. At Precision@20, the baseline performed better: 75.0% versus 72.0%.

The observed results do not show that machine learning beats the fixed rule. Both methods provided directional ranking enrichment on held-out clients from the same period, but the simpler baseline remains the preferred decision-support method. These measurements do not prove future-period performance, explain why traffic changed, or show that refreshing a recommended page will cause recovery.

---



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.